In [25]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [26]:
texts = [
    # Positive
    "I love this movie",
    "Excellent story",
    "I love chicken",
    "This is amazing and wonderful",
    "Best experience ever",
    "Absolutely fantastic work",


    # Negative
    "I hate this movie",
    "Mcdonalds is bad",
    "Terrible service and rude staff",
    "This is the worst thing ever",
    "Awful experience never again",
    "I really dislike this product",
]

In [27]:
#Lableling 

labels = np.array([1, 1, 1, 1, 1, 1, 
                   0, 0, 0, 0, 0, 0])

In [28]:
vocab_size = 1000
max_length = 6

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>") # oov stands for out of vocabulary, it will be used to represent words that are not in the vocabulary
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

print("Word Index:")
print(tokenizer.word_index)

print("\nInput Sequences:")
print(X)


Word Index:
{'<OOV>': 1, 'this': 2, 'i': 3, 'is': 4, 'love': 5, 'movie': 6, 'and': 7, 'experience': 8, 'ever': 9, 'excellent': 10, 'story': 11, 'chicken': 12, 'amazing': 13, 'wonderful': 14, 'best': 15, 'absolutely': 16, 'fantastic': 17, 'work': 18, 'hate': 19, 'mcdonalds': 20, 'bad': 21, 'terrible': 22, 'service': 23, 'rude': 24, 'staff': 25, 'the': 26, 'worst': 27, 'thing': 28, 'awful': 29, 'never': 30, 'again': 31, 'really': 32, 'dislike': 33, 'product': 34}

Input Sequences:
[[ 3  5  2  6  0  0]
 [10 11  0  0  0  0]
 [ 3  5 12  0  0  0]
 [ 2  4 13  7 14  0]
 [15  8  9  0  0  0]
 [16 17 18  0  0  0]
 [ 3 19  2  6  0  0]
 [20  4 21  0  0  0]
 [22 23  7 24 25  0]
 [ 2  4 26 27 28  9]
 [29  8 30 31  0  0]
 [ 3 32 33  2 34  0]]


In [29]:
#Position Embedding

class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [30]:
#Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
 
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [58]:
#build the model
embed_dim = 64

num_heads = 8

ff_dim = 32

inputs = layers.Input(shape=(max_length,))

 

x = TokenAndPositionEmbedding(

    max_length=max_length,

    vocab_size=vocab_size,

    embed_dim=embed_dim

)(inputs)

 

x = TransformerBlock(

    embed_dim=embed_dim,

    num_heads=num_heads,

    ff_dim=ff_dim

)(x)

 

x = layers.GlobalAveragePooling1D()(x)

 

outputs = layers.Dense(1, activation="sigmoid")(x)

 

model = tf.keras.Model(inputs=inputs, outputs=outputs)




In [59]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)     │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_7  │ (None, 6, 64)          │        64,384 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ (None, 6, 64)          │       137,120 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_7      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201,569 (787.38 KB)

 Trainable params: 201,569 (787.38 KB)

 Non-trainable params: 0 (0.00 B)

In [60]:
model.fit(
    X,
    labels,
    epochs=10,
    batch_size=2,
    verbose=1
)



Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5000 - loss: 1.0848
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5833 - loss: 0.7504 
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6667 - loss: 0.5088 
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7500 - loss: 0.4386 
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9167 - loss: 0.3533 
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8333 - loss: 0.2633 
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9167 - loss: 0.2203 
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.1466 
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0933 
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0702 


In [57]:
test_sentences = [
    "one of my favourite movies for real",
    "such a boring movie"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
one of my favourite movies for real -> 0.16241606
Prediction: Negative
such a boring movie -> 0.35904637
Prediction: Negative
